# KZA: Building Statistical Models for Football Betting

## Imports

In [63]:
import polars as pl
import importlib

## Introduction

### Objective
Our primary goal is to generate a profitable trading model betting on European Football matches by identifying discrepancies between our calculated "True Probability" and the odds or "Implied Probabilities" offered by bookmakers.

### The Pipeline
Our approach follows a structured data science lifecycle:

- **Research**: Investigate if there are market inefficiencies and look at potiental models.
- **Data Collection**: Sourcing online match and market data.
- **Exploratory Data Analysis**: Check if our data is valid and look for insights 
- **Feature Engineering**: Transforming raw stats into predictive signals.
- **Modelling**: Training and validating via time-series cross-validation.
- **Signal Generation**: Converting probabilities into bets.
- **Risk Management**: Deciding how much to stake on each bet.
- **Evaluation**: Assessing performance of our strategy the Closing Line.
- **Launch**: Deployment and monitoring.

## Research

### Evidence of Market Inefficiency

The Efficient Market Hypothesis suggests that in a highly liquid market, like European Football, the odds offered by bookmakers should reflect all available information, rendering it incredibly difficult to achieve a consistent edge. However when lopsided betting occurs, driven by fan bias toward "Big Six" clubs (e.g., Real Madrid, Manchester United) or "longshot" underdogs, bookmakers often adjust their odds away from the "True Probability" to encourage betting on the other side and reduce their risk.

We hypothesize that value exists in the "drift" between the market price and statistical reality, when the odds drift too far from the "True Probabilities". Some studies that support this idea are:

**TODO** Add papers

### Model Selection

To capture the "drift" between market prices and statistical reality, we chose a diverse set of three distinct modeling approaches.

**Simplest Model: Ordered Logistic Regression**

**Linear Baseline: Multimodal Logistic Regression**

**Non-Linear Logic: Gradient Boosted Trees (XGBoost)**

**TODO** Add papers

### Predicting Bookmaker Errors Directly

**TODO** Add papers

## Data Collection

To build a robust representation of European football, we integrated three distinct data dimensions: performance stats, historical rankings, and financial valuation.

### Data Sources
We aggregated our dataset from three primary pillars found on Kaggle and specialized football databases:

- Football-Data.co.uk: Our primary source for historical match results, betting odds, and match-level statistics (shots, corners, fouls).
- ClubELO: Used to integrate long-term team strength ratings. ELO provides a "memory" of a team's quality that persists beyond a single result.
- Transfermarkt (Market Values): We pulled squad valuations to act as a proxy for raw talent and depth. This helps the model distinguish between a "lucky" mid-table team and a powerhouse underperforming its budget.

### League Selection: The "Main 10"
To ensure data consistency and high liquidity for our betting strategy, we focused on the top 10 European leagues. This selection provides a massive sample size of roughly 3,800 matches per season, ensuring our model has enough "experience" to learn league-specific nuances.

Leagues Included:
- Premier League (England)
- La Liga (Spain), 
- Bundesliga (Germany)
- Serie A (Italy)
- Ligue 1 (France) 
- Primeira Liga (Portugal)
- Super League (Greece)
- Süper Lig (Turkey)
- Premiership (Scotland)
- Jupiler Pro League (Belgium)

### Time Frame
We opted for a 7-year historical window. This timeframe represents a good middle ground for sports modeling:

Relevance: Data older than 7 years may reflect a different era of tactical play (pre-heavy pressing/VAR), which can "pollute" modern predictions.
Volume: Six years provides over 22,000 matches, offering a great sample size to train and test our models without immediate overfitting.

### Implementation
To ensure reproducibility and clean execution, the entire ingestion pipeline—including cleaning, joining, and handling missing ELO/Value data—is encapsulated in a custom modular function. This pipeline handles the merging of disparate CSV files into a single, analysis-ready DataFrame.

Note: The full implementation of our data pipeline, including the load_data() logic, is available in our GitHub repository linked in the appendix.


In [64]:
import logging
from pipeline.data_loader import load_data
from utils.constants import MAIN_EUROPEAN_LEAGUES


logging.getLogger("pipeline").setLevel(logging.ERROR)

NUM_YEARS = 7
df = load_data(leagues=MAIN_EUROPEAN_LEAGUES, num_years=NUM_YEARS)

c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\pipeline\loaders\transfer_data.py:108: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  market_value_df = lineups_df.join_asof(
c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\pipeline\data_loader.py:112: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  combined_df = combined_df.join_asof(
c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\pipeline\data_loader.py:128: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  combined_df = combined_df.join_asof(


In [65]:
# Remove non numerical columns
df.drop("Datetime", "League", "HomeTeam", "AwayTeam", "FTR").describe()

statistic,FTHG,FTAG,HS,AS,HST,AST,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,home_mean_market_val,away_mean_market_val,home_team_elo,away_team_elo,home_team_elo_after,away_team_elo_after
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0,23208.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",1.534385,1.252671,13.329671,11.073164,4.807351,3.997027,2.993011,4.340993,4.851351,2.808971,4.078143,4.37512,9.1400e6,9.1107e6,1575.650804,1575.601178,1575.559799,1575.689047
"""std""",1.303884,1.173462,5.478007,4.898732,2.631926,2.383104,2.313294,1.49908,4.700387,1.965889,1.256326,3.681807,1.2821e7,1.2907e7,181.136821,181.145792,181.215731,181.097048
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,1.05,2.2,1.08,1.03,2.09,1.06,100000.0,100000.0,1130.909668,1127.796631,1127.796631,1133.155518
"""25%""",1.0,0.0,9.0,8.0,3.0,2.0,1.75,3.53,2.4,1.7,3.37,2.3,1.1864e6,1.18e6,1429.105103,1429.665649,1429.212402,1429.265991
"""50%""",1.0,1.0,13.0,11.0,4.0,4.0,2.32,3.84,3.4,2.24,3.65,3.23,4.1545e6,4.1045e6,1587.108643,1587.709351,1587.566162,1587.402466
"""75%""",2.0,2.0,17.0,14.0,6.0,5.0,3.22,4.5,5.25,3.07,4.28,4.86,1.1745e7,1.1636e7,1705.967773,1705.474854,1705.824463,1705.931396
"""max""",9.0,13.0,46.0,45.0,31.0,23.0,41.0,25.0,67.0,28.96,16.44,41.22,1.5e8,1.55e8,2088.261963,2090.13623,2090.13623,2091.465576


Looking at the dataset we can conclude that the data doesn't contain any red flags. All the mean values fall in a expected range and the max and min values aren't impossible. Even basic facts like home field advantage are here since mean FTHG (Full Time Home Goals) are greater than the mean FTAG.

## Feature Engineering

Sometimes the model fails because we haven't used the right set of features. Here we use a three step process to iteratively create, transform, and select features until we have list that may process an edge. 

First we need to add the specific season of each match to our dataframe. We will assume a new season starts earliest in July (Month 7). Then we will add a unique match id to each row for reference later

In [ ]:
from pipeline.feature_engineering import add_seasons

SEASON_START_MONTH = 7
df = add_seasons(df, SEASON_START_MONTH)
df = df.with_row_index(name="match_id")
df

match_id,Datetime,League,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,home_mean_market_val,away_mean_market_val,home_team_elo,away_team_elo,home_team_elo_after,away_team_elo_after,Season
u32,datetime[μs],str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0,2019-07-26 19:30:00,"""BEL""","""Genk""","""Kortrijk""","""H""",2,1,10,8,2,4,1.41,5.25,8.5,1.38,4.94,7.42,6.5364e6,790909.090909,1641.44104,1508.402954,1645.038574,1504.805298,"""2019/2020"""
1,2019-07-27 17:00:00,"""BEL""","""Cercle Brugge""","""Standard""","""A""",0,2,13,14,5,9,4.1,3.8,1.99,3.72,3.59,1.93,1.2909e6,2.6556e6,1306.84729,1563.234009,1302.001343,1568.079956,"""2019/2020"""
2,2019-07-27 19:00:00,"""BEL""","""St Truiden""","""Mouscron""","""A""",0,1,10,10,4,6,2.01,3.75,4.1,1.95,3.49,3.77,863636.363636,463636.363636,1482.195435,1404.303955,1470.709839,1415.789551,"""2019/2020"""
3,2019-07-27 19:00:00,"""BEL""","""Waregem""","""Mechelen""","""A""",0,2,7,10,2,5,2.37,3.76,3.15,2.2,3.58,3.03,895454.545455,713636.363636,1444.63208,1341.168579,1427.779663,1358.020874,"""2019/2020"""
4,2019-07-27 19:30:00,"""BEL""","""Waasland-Beveren""","""Club Brugge""","""A""",1,3,7,25,2,22,6.5,4.6,1.59,5.53,4.36,1.53,627272.727273,5.5818e6,1365.98999,1637.778564,1361.562378,1642.206055,"""2019/2020"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
23203,2026-03-09 17:00:00,"""TUR""","""Alanyaspor""","""Genclerbirligi""","""D""",0,0,18,3,7,0,1.91,3.5,4.2,1.83,3.43,3.94,909090.909091,1.4091e6,1404.931519,1343.604614,1401.668823,1346.86731,"""2025/2026"""
23204,2026-03-09 17:00:00,"""TUR""","""Kayserispor""","""Trabzonspor""","""A""",1,3,14,18,6,5,3.65,3.7,2.0,3.46,3.49,1.94,1.41e6,6.1364e6,1352.259033,1540.121582,1345.526978,1546.85376,"""2025/2026"""
23205,2026-03-09 19:45:00,"""ITA""","""Lazio""","""Sassuolo""","""H""",2,1,13,7,5,4,2.2,3.3,3.55,2.16,3.18,3.42,1.05e7,9.4364e6,1686.83728,1637.046021,1692.672363,1631.21106,"""2025/2026"""


### Creation

To have a wide range of potiental features to use in our model we have decided to create a comprehensive list of custom caclulated metrics that cover a variety of areas in football.

The features are:
- Elo Difference
- Market Value of Starting 11
- Elo per Market Value Difference
- Rolling average of goals
- Rolling average of conversion rate (goals / shots on target)
- Rolling average of 

In [13]:
df = df.with_columns(
    (pl.col("home_team_elo") - pl.col("away_team_elo")).alias("elo_diff"),
    pl.col("home_mean_market_val").log().alias("HMV_log_val"),
    pl.col("away_mean_market_val").log().alias("AMV_log_val")
)

df = df.with_columns(
    (pl.col("HMV_log_val") - pl.col("AMV_log_val")).alias("MV_log_diff"),
    (
        pl.col("home_team_elo") / pl.col("HMV_log_val")
        - pl.col("away_team_elo") / pl.col("AMV_log_val")
    ).alias("elo_per_value_diff")
).drop("HMV_log_val", "AMV_log_val")

df.select(["Datetime", "elo_diff", "MV_log_diff", "elo_per_value_diff"])

Datetime,elo_diff,MV_log_diff,elo_per_value_diff
datetime[μs],f64,f64,f64
2019-07-26 19:30:00,133.038086,2.111953,-6.469911
2019-07-27 17:00:00,-256.386719,-0.721307,-12.803702
2019-07-27 19:00:00,77.891479,0.622051,0.800117
2019-07-27 19:00:00,103.463501,0.226958,5.901429
2019-07-27 19:30:00,-271.788574,-2.185888,-3.096941
…,…,…,…
2026-03-09 17:00:00,61.326904,-0.438255,7.501077
2026-03-09 17:00:00,-187.862549,-1.470643,-3.033283
2026-03-09 19:45:00,49.79126,0.106805,2.406425


In [14]:
from pipeline import feature_engineering
importlib.reload(feature_engineering)
from pipeline.feature_engineering import add_custom_ewm_features

SHORT_WINDOW = 4
MEDIUM_WINDOW = 15
LONG_WINDOW = 30
windows = [SHORT_WINDOW, MEDIUM_WINDOW, LONG_WINDOW]

# Calculate Global Priors
# SOT Ratio (Total SOT / Total Shots)
stats = df.select([
    (pl.col("HST").sum() + pl.col("AST").sum()).alias("total_sot"),
    (pl.col("HS").sum() + pl.col("AS").sum()).alias("total_shots"),
    (pl.col("FTHG").sum() + pl.col("FTAG").sum()).alias("total_goals")
])
prior_sot = stats.get_column("total_sot")[0] / stats.get_column("total_shots")[0]
prior_conv = stats.get_column("total_goals")[0] / stats.get_column("total_sot")[0]
prior_save = 1 - prior_conv  # Simplified: Saves are just non-goals on target

# Bayesian Smoothing Constants
ALPHA_SHOTS = 5
ALPHA_GOALS = 5

ewm_features = {
    "goals": {
        "home": "FTHG",
        "away": "FTAG"
    },
    # Shots on Target Ratio (Smoothed)
    "sot_ratio": {
        "home": (pl.col("HST") + (ALPHA_SHOTS * prior_sot)) / (pl.col("HS") + ALPHA_SHOTS),
        "away": (pl.col("AST") + (ALPHA_SHOTS * prior_sot)) / (pl.col("AS") + ALPHA_SHOTS)
    },
    # SOT Against Ratio (Smoothed)
    "sot_a_ratio": {
        "home": (pl.col("AST") + (ALPHA_SHOTS * prior_sot)) / (pl.col("AS") + ALPHA_SHOTS),
        "away": (pl.col("HST") + (ALPHA_SHOTS * prior_sot)) / (pl.col("HS") + ALPHA_SHOTS)
    },
    # Conversion Rate (Smoothed)
    "conversion_rate": {
        "home": (pl.col("FTHG") + (ALPHA_GOALS * prior_conv)) / (pl.col("HST") + ALPHA_GOALS),
        "away": (pl.col("FTAG") + (ALPHA_GOALS * prior_conv)) / (pl.col("AST") + ALPHA_GOALS)
    },
    # Save Rate (Smoothed)
    "save_rate": {
        "home": ((pl.col("AST") - pl.col("FTAG")) + (ALPHA_GOALS * prior_save)) / (pl.col("AST") + ALPHA_GOALS),
        "away": ((pl.col("HST") - pl.col("FTHG")) + (ALPHA_GOALS * prior_save)) / (pl.col("HST") + ALPHA_GOALS)
    },
    "elo_adv": {
        "home": pl.col("elo_diff"),
        "away": -pl.col("elo_diff")
    }
}

df, latest_stats = add_custom_ewm_features(df, windows, ewm_features)

df = df.drop(
    f"home_elo_adv_ewm_{MEDIUM_WINDOW}", f"home_elo_adv_ewm_{LONG_WINDOW}",
    f"away_elo_adv_ewm_{MEDIUM_WINDOW}", f"away_elo_adv_ewm_{LONG_WINDOW}"
).drop_nulls()

latest_stats.filter(pl.col("team") == "Man City")

team,match_id,Datetime,goals,sot_ratio,sot_a_ratio,conversion_rate,save_rate,elo_adv,goals_ewm_4,goals_ewm_15,goals_ewm_30,sot_ratio_ewm_4,sot_ratio_ewm_15,sot_ratio_ewm_30,sot_a_ratio_ewm_4,sot_a_ratio_ewm_15,sot_a_ratio_ewm_30,conversion_rate_ewm_4,conversion_rate_ewm_15,conversion_rate_ewm_30,save_rate_ewm_4,save_rate_ewm_15,save_rate_ewm_30,elo_adv_ewm_4,elo_adv_ewm_15,elo_adv_ewm_30
str,u32,datetime[μs],i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Man City""",23124,2026-03-04 19:30:00,2,0.338614,0.414569,0.298564,0.601915,209.370361,1.810772,1.854954,1.933117,0.360225,0.355931,0.361307,0.351489,0.35391,0.354501,0.308499,0.33455,0.341203,0.700168,0.715499,0.709508,174.221082,166.893454,169.955921


In [15]:
momentum_features = ["goals", "sot_ratio", "sot_a_ratio", "conversion_rate", "save_rate"]

for feature in momentum_features:
    for side in ("home", "away"):
        df = df.with_columns(
            (pl.col(f"{side}_{feature}_ewm_{SHORT_WINDOW}")
            - pl.col(f"{side}_{feature}_ewm_{LONG_WINDOW}"))
            .alias(f"{side}_{feature}_momentum")
        )

df

match_id,Datetime,League,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,home_mean_market_val,away_mean_market_val,home_team_elo,away_team_elo,home_team_elo_after,away_team_elo_after,Season,elo_diff,MV_log_diff,elo_per_value_diff,home_goals_ewm_4,home_goals_ewm_15,home_goals_ewm_30,home_sot_ratio_ewm_4,home_sot_ratio_ewm_15,home_sot_ratio_ewm_30,home_sot_a_ratio_ewm_4,home_sot_a_ratio_ewm_15,home_sot_a_ratio_ewm_30,home_conversion_rate_ewm_4,home_conversion_rate_ewm_15,home_conversion_rate_ewm_30,home_save_rate_ewm_4,home_save_rate_ewm_15,home_save_rate_ewm_30,home_elo_adv_ewm_4,away_goals_ewm_4,away_goals_ewm_15,away_goals_ewm_30,away_sot_ratio_ewm_4,away_sot_ratio_ewm_15,away_sot_ratio_ewm_30,away_sot_a_ratio_ewm_4,away_sot_a_ratio_ewm_15,away_sot_a_ratio_ewm_30,away_conversion_rate_ewm_4,away_conversion_rate_ewm_15,away_conversion_rate_ewm_30,away_save_rate_ewm_4,away_save_rate_ewm_15,away_save_rate_ewm_30,away_elo_adv_ewm_4,home_goals_momentum,away_goals_momentum,home_sot_ratio_momentum,away_sot_ratio_momentum,home_sot_a_ratio_momentum,away_sot_a_ratio_momentum,home_conversion_rate_momentum,away_conversion_rate_momentum,home_save_rate_momentum,away_save_rate_momentum
u32,datetime[μs],str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
9,2019-08-02 19:30:00,"""BEL""","""Club Brugge""","""St Truiden""","""H""",6,0,16,4,10,2,1.26,6.5,12.0,1.24,6.04,10.99,4.8091e6,677272.727273,1642.0354,1470.539185,1649.497559,1463.077026,"""2019/2020""",171.496216,1.960189,-2.808036,3.0,3.0,3.0,0.793466,0.793466,0.793466,0.316997,0.316997,0.316997,0.169732,0.169732,0.169732,0.631033,0.631033,0.631033,271.788574,0.0,0.0,0.0,0.386931,0.386931,0.386931,0.520264,0.520264,0.520264,0.175863,0.175863,0.175863,0.765203,0.765203,0.765203,77.891479,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14,2019-08-03 17:00:00,"""BEL""","""Standard""","""Waregem""","""H""",4,0,16,3,7,1,1.54,4.89,6.25,1.49,4.53,5.8,2.8778e6,954545.454545,1567.909302,1427.609009,1574.792847,1420.725464,"""2019/2020""",140.300293,1.103538,1.74027,2.0,2.0,2.0,0.56863,0.56863,0.56863,0.377998,0.377998,0.377998,0.255912,0.255912,0.255912,0.841723,0.841723,0.841723,256.386719,0.0,0.0,0.0,0.316997,0.316997,0.316997,0.453598,0.453598,0.453598,0.22611,0.22611,0.22611,0.641723,0.641723,0.641723,103.463501,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,2019-08-03 19:00:00,"""BEL""","""Kortrijk""","""Charleroi""","""D""",1,1,7,11,4,6,2.44,3.61,3.05,2.3,3.43,2.95,845454.545455,1.4591e6,1504.634644,1507.21167,1503.128906,1508.717529,"""2019/2020""",-2.577026,-0.545694,4.057198,1.0,1.0,1.0,0.446459,0.446459,0.446459,0.253598,0.253598,0.253598,0.286974,0.286974,0.286974,0.488176,0.488176,0.488176,-133.038086,1.0,1.0,1.0,0.483664,0.483664,0.483664,0.580397,0.580397,0.580397,0.286974,0.286974,0.286974,0.713026,0.713026,0.713026,-39.523804,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
18,2019-08-03 19:00:00,"""BEL""","""Oostende""","""Cercle Brugge""","""H""",3,1,11,9,7,4,2.3,3.75,3.27,2.21,3.56,3.01,1.2955e6,1.505e6,1388.540405,1301.830688,1394.955811,1295.415283,"""2019/2020""",86.709717,-0.149931,7.135783,2.0,2.0,2.0,0.600305,0.600305,0.600305,0.390198,0.390198,0.390198,0.325706,0.325706,0.325706,0.765203,0.765203,0.765203,-126.482544,0.0,0.0,0.0,0.377998,0.377998,0.377998,0.56863,0.56863,0.56863,0.158277,0.158277,0.158277,0.744088,0.744088,0.744088,-256.386719,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,2019-08-03 19:30:00,"""BEL""","""Mechelen""","""Genk""","""H""",3,1,11,9,5,5,4.33,3.85,1.88,4.06,3.68,1.83,713636.363636,6.4273e6,1357.85022,1644.86792,1377.158203,1625.559937,"""2019/2020""",-287.0177,-2.197932,-4.183938,2.0,2.0,2.0,0.453598,0.453598,0.453598,0.316997,0.316997,0.316997,0.358277,0.358277,0.358277,0.77389,0.77389,0.77389,-103.463501,2.0,2.0,2.0,0.253598

In [16]:
for side in ("home", "away"):
    df = df.with_columns(
        (
            pl.col(f"{side}_conversion_rate_ewm_{SHORT_WINDOW}") 
            + pl.col(f"{side}_save_rate_ewm_{SHORT_WINDOW}")
        ).alias(f"{side}_luck_factor")
    )

In [17]:
import polars.selectors as cs
# machine learning featurs
training_df = df.filter(pl.col("Season") != "2019/2020")

ml_features = training_df.drop(
    "HomeTeam", "AwayTeam", "FTR", "FTHG", "FTAG", "HS", "AS", "HST", "AST", "MaxH", "MaxD", "MaxA", "AvgH", "AvgD", "AvgA"
)
possion_features = training_df.select(["match_id", "Datetime", "Season", "HomeTeam", "AwayTeam"])

ml_results = training_df.select(["match_id", "Season", "Datetime", "FTR"])
possion_results = training_df.select(["match_id", "Datetime", "FTHG", "FTAG"])

print("Machine Learning Features:")
for col in ml_features.columns:
    if col not in ("match_id", "Datetime"):
        print(f"- {col}")

print("\nPossion Features:")
for col in ml_features.columns:
    print(f"- {col}")

Machine Learning Features:
- League
- home_mean_market_val
- away_mean_market_val
- home_team_elo
- away_team_elo
- home_team_elo_after
- away_team_elo_after
- Season
- elo_diff
- MV_log_diff
- elo_per_value_diff
- home_goals_ewm_4
- home_goals_ewm_15
- home_goals_ewm_30
- home_sot_ratio_ewm_4
- home_sot_ratio_ewm_15
- home_sot_ratio_ewm_30
- home_sot_a_ratio_ewm_4
- home_sot_a_ratio_ewm_15
- home_sot_a_ratio_ewm_30
- home_conversion_rate_ewm_4
- home_conversion_rate_ewm_15
- home_conversion_rate_ewm_30
- home_save_rate_ewm_4
- home_save_rate_ewm_15
- home_save_rate_ewm_30
- home_elo_adv_ewm_4
- away_goals_ewm_4
- away_goals_ewm_15
- away_goals_ewm_30
- away_sot_ratio_ewm_4
- away_sot_ratio_ewm_15
- away_sot_ratio_ewm_30
- away_sot_a_ratio_ewm_4
- away_sot_a_ratio_ewm_15
- away_sot_a_ratio_ewm_30
- away_conversion_rate_ewm_4
- away_conversion_rate_ewm_15
- away_conversion_rate_ewm_30
- away_save_rate_ewm_4
- away_save_rate_ewm_15
- away_save_rate_ewm_30
- away_elo_adv_ewm_4
- home_goal

In [18]:
import polars as pl
import plotly.express as px

def plot_corr_matrix(features: pl.DataFrame, size):
    numeric_df = features.select(cs.numeric())

    corr_matrix = numeric_df.corr()

    # 3. Create the Plotly Heatmap
    fig = px.imshow(
        corr_matrix.to_numpy(),
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        color_continuous_scale='RdBu_r', # Red-Blue scale (standard for corr)
        zmin=-1, zmax=1,                 # Correlation is always -1 to 1
        title="Feature Correlation Matrix",
        labels=dict(color="Pearson Corr"),
        aspect="auto"
    )

    fig.update_layout(
        width=size[0],
        height=size[1],
        xaxis_tickangle=-45
    )
    fig.show()

plot_corr_matrix(ml_features, size=(1200, 1200))

In [19]:

ml_features = (
    ml_features
    .rename({
        f"home_elo_adv_ewm_{SHORT_WINDOW}": "home_elo_adv",
        f"away_elo_adv_ewm_{SHORT_WINDOW}": "away_elo_adv"
    })
)

ml_features = ml_features.drop([
    feature 
    for feature in ml_features.columns 
    if (f"ewm_{MEDIUM_WINDOW}" in feature) or (f"ewm_{SHORT_WINDOW}" in feature)
])

for side in ("home", "away"):
    ml_features = ml_features.drop(
        f"{side}_luck_factor", 
        f"{side}_mean_market_val",
        f"{side}_team_elo"
    )

ml_features = ml_features.drop("MV_log_diff")
plot_corr_matrix(ml_features, size=(900, 900))

In [20]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import polars as pl
import pandas as pd

def calculate_vif(features: pl.DataFrame):
    numeric_df = features.select(cs.numeric()).drop("match_id")
    # 2. Convert to Pandas (statsmodels requires it)
    X = numeric_df.to_pandas()
    
    # 3. Calculate VIF for each feature
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X.values, i) 
        for i in range(len(X.columns))
    ]
    
    return vif_data.sort_values("VIF", ascending=False)

# Run it
vif_results = pl.DataFrame(calculate_vif(ml_features))
for row in vif_results.iter_rows(named=True):
    print(f"Feature: {row['feature']}, VIF: {row['VIF']}")

Feature: away_team_elo_after, VIF: 10071.621017907206
Feature: home_team_elo_after, VIF: 10059.405881969897
Feature: away_save_rate_ewm_30, VIF: 832.5243588705604
Feature: home_save_rate_ewm_30, VIF: 828.1287194687652
Feature: home_conversion_rate_ewm_30, VIF: 433.9886593630506
Feature: away_conversion_rate_ewm_30, VIF: 431.7612099217872
Feature: home_sot_ratio_ewm_30, VIF: 302.5567013967623
Feature: away_sot_ratio_ewm_30, VIF: 301.4618731263347
Feature: home_sot_a_ratio_ewm_30, VIF: 233.76800444864583
Feature: away_sot_a_ratio_ewm_30, VIF: 229.7397117204349
Feature: elo_diff, VIF: 122.07214771187404
Feature: away_goals_ewm_30, VIF: 84.10228462964322
Feature: home_goals_ewm_30, VIF: 83.66274022342404
Feature: away_elo_adv, VIF: 4.777741529952219
Feature: home_elo_adv, VIF: 4.752299302957902
Feature: away_goals_momentum, VIF: 4.285179346015365
Feature: home_goals_momentum, VIF: 4.189570657025174
Feature: away_conversion_rate_momentum, VIF: 3.4737511339252993
Feature: home_conversion_rat

In [21]:
residuals = training_df.select(
    "match_id", "Datetime", "Season", "FTR", "AvgH", "AvgA", "AvgD"
)

residuals = residuals.with_columns(
    AvgH= 1 / pl.col("AvgH"),
    AvgA= 1 / pl.col("AvgA"),
    AvgD= 1 / pl.col("AvgD")
)

residuals = residuals.with_columns(implied_probs_sum=pl.col("AvgH") + pl.col("AvgD") + pl.col("AvgA"))
residuals = residuals.with_columns(
    pl.col(["AvgH", "AvgD", "AvgA"]) / pl.col("implied_probs_sum")
).rename({
    "AvgH": "home_prob",
    "AvgA": "away_prob",
    "AvgD": "draw_prob"
}).drop("implied_probs_sum")

residuals = residuals.to_dummies("FTR")
residuals = residuals.with_columns(
    home_residual=pl.col("home_prob") - pl.col("FTR_H"),
    away_residual=pl.col("away_prob") - pl.col("FTR_A"),
    draw_residual=pl.col("draw_prob") - pl.col("FTR_D"),
).drop(cs.matches(r"_prob$") | cs.matches("^FTR"))
residuals

match_id,Datetime,Season,home_residual,away_residual,draw_residual
u32,datetime[μs],str,f64,f64,f64
2846,2020-07-01 17:15:00,"""2020/2021""",-0.427031,0.178846,0.248185
2847,2020-07-01 18:00:00,"""2020/2021""",-0.33608,0.132233,0.203847
2848,2020-07-01 18:00:00,"""2020/2021""",0.398858,-0.701788,0.30293
2849,2020-07-01 18:00:00,"""2020/2021""",-0.622677,0.330483,0.292195
2850,2020-07-01 18:30:00,"""2020/2021""",0.407546,-0.724376,0.31683
…,…,…,…,…,…
23203,2026-03-09 17:00:00,"""2025/2026""",0.500502,0.232467,-0.732968
23204,2026-03-09 17:00:00,"""2025/2026""",0.264907,-0.527537,0.26263
23205,2026-03-09 19:45:00,"""2025/2026""",-0.567254,0.273313,0.293941


In [22]:
from statsmodels.multivariate.manova import MANOVA

manov_df = residuals.join(
    ml_features,
    on=['match_id', 'Datetime'],
    how='left'
)

manov_df = manov_df.select(cs.numeric()).drop("match_id")

feature_list = ml_features.select(cs.numeric()).drop("match_id").columns
# 2. Join them with a plus sign
features_formula = " + ".join(feature_list)

ma = MANOVA.from_formula(f'home_residual + away_residual ~ {features_formula}', data=manov_df.to_pandas())
print(ma.mv_test())

                      Multivariate linear model
                                                                      
-----------------------------------------------------------------------
          Intercept         Value   Num DF    Den DF    F Value  Pr > F
-----------------------------------------------------------------------
             Wilks' lambda  0.9997  2.0000  20242.0000   2.8988  0.0551
            Pillai's trace  0.0003  2.0000  20242.0000   2.8988  0.0551
    Hotelling-Lawley trace  0.0003  2.0000  20242.0000   2.8988  0.0551
       Roy's greatest root  0.0003  2.0000  20242.0000   2.8988  0.0551
----------------------------------------------------------------------
                                                                      
----------------------------------------------------------------------
    home_team_elo_after    Value  Num DF   Den DF     F Value   Pr > F
----------------------------------------------------------------------
            Wilks' lam

In [23]:
import polars as pl
from scipy import stats


manov_df = ml_results.join(
    ml_features,
    on=['match_id', 'Datetime'],
    how='left'
)

features = manov_df.select(cs.numeric()).drop("match_id").columns
target_col = "FTR"

results = {}

for feature in features:
    # 2. Group data by category and collect the numerical values into lists
    groups = (
        manov_df.group_by(target_col)
        .agg(pl.col(feature))
        .get_column(feature)
        .to_list()
    )
    
    # 3. Perform One-Way ANOVA (*groups unpacks the list of arrays)
    f_stat, p_val = stats.f_oneway(*groups)
    
    results[feature] = {"F-Statistic": f_stat, "P-Value": p_val}

# 4. View results as a summary table
summary_df = pl.DataFrame([
    {"feature": k, "f_stat": v["F-Statistic"], "p_value": v["P-Value"]} 
    for k, v in results.items()
]).sort("p_value")

for row in summary_df.iter_rows(named=True):
    print(row["feature"], round(row["f_stat"], 2), round(row['p_value'], 2))

elo_diff 2155.9 0.0
home_elo_adv 757.35 0.0
away_goals_ewm_30 753.87 0.0
away_elo_adv 749.3 0.0
home_goals_ewm_30 729.83 0.0
away_team_elo_after 578.54 0.0
home_team_elo_after 573.45 0.0
elo_per_value_diff 477.17 0.0
away_conversion_rate_ewm_30 210.11 0.0
home_conversion_rate_ewm_30 196.36 0.0
away_sot_ratio_ewm_30 124.97 0.0
home_sot_ratio_ewm_30 84.88 0.0
home_save_rate_ewm_30 68.61 0.0
away_save_rate_ewm_30 65.1 0.0
home_sot_a_ratio_ewm_30 15.58 0.0
away_sot_a_ratio_ewm_30 11.65 0.0
away_sot_ratio_momentum 2.67 0.07
away_save_rate_momentum 0.69 0.5
home_save_rate_momentum 0.58 0.56
home_goals_momentum 0.41 0.66
home_conversion_rate_momentum 0.4 0.67
away_goals_momentum 0.37 0.69
home_sot_ratio_momentum 0.35 0.71
away_conversion_rate_momentum 0.16 0.85
away_sot_a_ratio_momentum 0.15 0.86
home_sot_a_ratio_momentum 0.13 0.87


In [24]:
import polars as pl

manov_df = residuals.join(
    ml_features,
    on=['match_id', 'Datetime'],
    how='left'
)

# 1. Convert to pandas for Plotly compatibility (zero-copy in Polars)
plot_df = manov_df.to_pandas()

feature = "away_sot_ratio_momentum"
# 2. Create the scatter plot with trendlines
fig = px.scatter(
    plot_df, 
    x=feature, 
    y="home_residual", 
    trendline="ols",  # Adds the linear regression lines
    title="Interactive Residuals vs. SOT Ratio Momentum",
    template="plotly_white",
)

# 3. Add a horizontal line at Zero (The 'Perfect Prediction' line)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.7)

# 4. Improve layout
fig.update_layout(
    xaxis_title=feature.replace("_", " ").capitalize(),
    yaxis_title="Residuals (Actual - Predicted)",
    legend_title="Match Result"
)

fig.show()

In [25]:
import polars as pl
from sklearn.feature_selection import mutual_info_regression

residual_cols = ["home_residual", "away_residual", "draw_residual"]
features = ml_features.select(cs.numeric()).drop("match_id").columns
X = ml_features.select(features).to_pandas()

mi_data = []
for feat in features:
    # Prepare X (2D array required by sklearn)
    X = ml_features.select(feat).to_pandas()
    
    row = {"feature": feat}
    for res in residual_cols:
        y = residuals.get_column(res).to_pandas()
        # Calculate MI (returns an array, we take the first element [0])
        score = mutual_info_regression(X, y, random_state=42)[0]
        row[res] = score
        
    mi_data.append(row)

# 4. Convert to Polars for easy viewing/sorting
mi_results_df = pl.DataFrame(mi_data)

In [26]:
for row in mi_results_df.sort('home_residual', descending=True).iter_rows(named=True):
    print(
        row["feature"], 
        round(row["home_residual"], 3), 
        round(row['away_residual'], 3), 
        round(row['draw_residual'], 3)
    )

elo_diff 1.161 1.158 0.589
home_elo_adv 0.275 0.254 0.162
home_goals_ewm_30 0.263 0.235 0.223
away_elo_adv 0.255 0.264 0.047
away_goals_ewm_30 0.234 0.272 0.098
away_team_elo_after 0.165 0.17 0.048
home_team_elo_after 0.163 0.155 0.105
elo_per_value_diff 0.161 0.171 0.052
home_conversion_rate_ewm_30 0.083 0.069 0.055
away_conversion_rate_ewm_30 0.075 0.093 0.036
home_sot_ratio_ewm_30 0.049 0.039 0.048
away_sot_ratio_ewm_30 0.045 0.055 0.022
home_save_rate_ewm_30 0.032 0.035 0.02
away_save_rate_ewm_30 0.027 0.02 0.007
home_sot_a_ratio_ewm_30 0.015 0.0 0.013
home_goals_momentum 0.01 0.019 0.021
away_save_rate_momentum 0.008 0.008 0.003
away_goals_momentum 0.007 0.015 0.012
away_sot_ratio_momentum 0.005 0.0 0.001
home_save_rate_momentum 0.0 0.0 0.004
home_sot_ratio_momentum 0.0 0.003 0.0
away_sot_a_ratio_ewm_30 0.0 0.005 0.005
home_sot_a_ratio_momentum 0.0 0.0 0.007
away_sot_a_ratio_momentum 0.0 0.004 0.004
home_conversion_rate_momentum 0.0 0.003 0.006
away_conversion_rate_momentum 0.0 0.

In [27]:
residual_selected_columns = [
    "elo_per_value_diff", "away_sot_ratio_momentum",
]

for side in ("home", "away"):
    residual_selected_columns.extend(
        [
           # f"{side}_elo_adv",
        ]
    )

outcome_selected_columns = [
    "elo_diff", "elo_per_value_diff"
]

for side in ("home", "away"):
    outcome_selected_columns.extend(
        [
            f"{side}_elo_adv",
        ]
    )

ml_residuals = (
    ml_features
    .select("match_id", "Datetime", "Season", "League", *residual_selected_columns,
            conversion_diff=(
                pl.col("home_conversion_rate_ewm_30") - pl.col("away_conversion_rate_ewm_30")
            ))
)
ml_outcomes = ml_features.select("match_id", "Datetime", "Season", *outcome_selected_columns, 
                                 goals_ewm_30_diff=(
                                     pl.col("home_goals_ewm_30") - pl.col("away_goals_ewm_30")
                                 ))

In [28]:
vif_results = pl.DataFrame(calculate_vif(ml_residuals))
for row in vif_results.iter_rows(named=True):
    print(f"Feature: {row['feature']}, VIF: {row['VIF']}")

Feature: elo_per_value_diff, VIF: 1.185611873307703
Feature: conversion_diff, VIF: 1.18534228308432
Feature: away_sot_ratio_momentum, VIF: 1.0002837238813742


In [29]:
probs = training_df.select(
    "match_id", "Datetime", "Season", "FTR", "AvgH", "AvgA", "AvgD"
)

probs = probs.with_columns(
    AvgH= 1 / pl.col("AvgH"),
    AvgA= 1 / pl.col("AvgA"),
    AvgD= 1 / pl.col("AvgD")
)

probs = probs.with_columns(implied_probs_sum=pl.col("AvgH") + pl.col("AvgD") + pl.col("AvgA"))
probs = probs.with_columns(
    pl.col(["AvgH", "AvgD", "AvgA"]) / pl.col("implied_probs_sum")
).rename({
    "AvgH": "home_prob",
    "AvgA": "away_prob",
    "AvgD": "draw_prob"
}).drop("implied_probs_sum", "FTR")

In [30]:
from pipeline.models.ordinal import OrdinalBettingWrapper
from pipeline.models.xgboost import MultiClassXGBoostWrapper
from pipeline import model
importlib.reload(model)
from pipeline.model import SequentialSeasonValidator
from pipeline.models import bayesian, multimodal
importlib.reload(bayesian)
importlib.reload(multimodal)


from pipeline.models.multimodal import MultiModalLogitWrapper
from pipeline.models.bayesian import BayesianLassoLogitWrapper

outcomes = training_df.select(
    "match_id", "Datetime", "Season", 
    outcome=pl.col("FTR").replace({"H": 2, "D": 1, "A": 0}).cast(pl.Int64)
)

model_df = outcomes.join(
    probs,
    on=("match_id", "Datetime", "Season"),
    how="left"
)

model_outcomes_df = model_df.join(
    ml_outcomes,
    on=("match_id", "Datetime", "Season"),
    how="left"
)

model_residual_df = model_df.join(
    ml_residuals,
    on=("match_id", "Datetime", "Season"),
    how="left"
)

params = {
    # The Essentials
    'max_depth': 3,             # Shallow trees to avoid overfitting noise
    'learning_rate': 0.1,      # High enough to learn, low enough to be stable
    
    # Regularization (Crucial for 2020 data)
    'min_child_weight': 5,      # Won't make a rule unless it hits ~5 matches
    'lambda': 1.5,              # L2 regularization (keeps weights small)
    'subsample': 0.8,           # Use 80% of matches per tree for variety
    
    # Technicals
    'tree_method': 'hist',      # Fast training
    'nthread': -1               # Use all CPU cores
}

outcomes_features = ml_outcomes.select(cs.numeric()).drop("match_id").columns
residuals_features = ml_residuals.select(cs.numeric()).drop("match_id").columns

outcomes_models = [
    OrdinalBettingWrapper(predict_residuals=False),
    MultiClassXGBoostWrapper(predict_residuals=False, params=params),
]

residual_models = [
    #OrdinalBettingWrapper(predict_residuals=True),
    #MultiClassXGBoostWrapper(predict_residuals=True, params=params),
    MultiModalLogitWrapper(lasso=False),
    MultiModalLogitWrapper(lasso=True),
    BayesianLassoLogitWrapper()
]

for res_model in residual_models:
    validator = SequentialSeasonValidator(
        model_wrapper=res_model,
        features=residuals_features,
        start_season="2020/2021",
        num_seasons=5,
        target_col="outcome"
    )


    print("Residual Model")
    print(validator.run(model_residual_df))

Residual Model
shape: (4, 5)
┌───────────────────┬────────────┬────────────┬─────────────┬─────────────┐
│ validation_season ┆ train_size ┆ model_loss ┆ bookie_loss ┆ improvement │
│ ---               ┆ ---        ┆ ---        ┆ ---         ┆ ---         │
│ str               ┆ i64        ┆ f64        ┆ f64         ┆ f64         │
╞═══════════════════╪════════════╪════════════╪═════════════╪═════════════╡
│ 2021/2022         ┆ 3906       ┆ 0.968743   ┆ 0.970579    ┆ 0.001836    │
│ 2022/2023         ┆ 7435       ┆ 0.95186    ┆ 0.952291    ┆ 0.000431    │
│ 2023/2024         ┆ 10909      ┆ 0.941259   ┆ 0.944828    ┆ 0.003569    │
│ 2024/2025         ┆ 14379      ┆ 0.9564     ┆ 0.958177    ┆ 0.001777    │
└───────────────────┴────────────┴────────────┴─────────────┴─────────────┘
Residual Model
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.9785650317693048
            Iterations: 92
            Function evaluations: 93
            Gradient ev

c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\DELL\Github Repos\model-sports-betting-icl-axioms-comp\venv\Lib\site-packages\sklearn\linear_model\_logistic.

In [31]:
final_model = residual_models[1]

validator = SequentialSeasonValidator(
    model_wrapper=final_model,
    features=residuals_features,
    start_season="2020/2021",
    num_seasons=4,
    target_col="outcome"
)

validator.fit_all_training_data(model_residual_df)
final_model.model_res.summary()

Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.9599079663111728
            Iterations: 119
            Function evaluations: 119
            Gradient evaluations: 119


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                outcome   No. Observations:                14379
Model:                        MNLogit   Df Residuals:                    14369
Method:                           MLE   Df Model:                            8
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                  0.1068
Time:                        23:36:54   Log-Likelihood:                -13797.
converged:                       True   LL-Null:                       -15446.
Covariance Type:            nonrobust   LLR p-value:                     0.000
===========================================================================================
              outcome=1       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                      -0.0608      0.025     -2.456      0.014      -0.109      -0.012
elo_per_value_diff         -0.0260      0.028     -0.936      0.349      -0.080       0.028
away_sot_ratio_momentum     0.0512      0.019      2.644      0.008       0.013       0.089
conversion_diff            -0.0319      0.027     -1.190      0.234      -0.084       0.021
mkt_logit_home              0.0052      0.095      0.055      0.956      -0.181       0.192
mkt_logit_draw              0.6435      0.088      7.338      0.000       0.472       0.815
-------------------------------------------------------------------------------------------
              outcome=2       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                       0.3711      0.023     16.360      0.000       0.327       0.416
elo_per_value_diff         -0.1005      0.026     -3.799      0.000      -0.152      -0.049
away_sot_ratio_momentum          0        nan        nan        nan         nan         nan
conversion_diff            -0.0530      0.026     -2.079      0.038      -0.103      -0.003
mkt_logit_home              1.3438      0.032     41.460      0.000       1.280       1.407
mkt_logit_draw                   0        nan        nan        nan         nan         nan
===========================================================================================
"""

In [32]:
USE_MODEL = True

test_seasons = [24, 25]
test_seasons = [f"20{season}/20{season + 1}" for season in test_seasons]

if USE_MODEL:
    # Your current model prediction logic
    preds = final_model.predict(
        model_residual_df.filter(pl.col("Season").is_in(test_seasons)).select(residuals_features), 
        market_probs=probs.filter(pl.col("Season").is_in(test_seasons))
    )
    val_probs = pl.DataFrame(preds, schema=["m_away_probs", "m_draw_probs", "m_home_probs"])
else:
    # Use the bookie's own probs (extracted from your 'probs' dataframe)
    # We rename them to match the schema used in the join
    val_probs = (
        probs.filter(pl.col("Season").is_in(test_seasons))
        .select([
            pl.col("away_prob").alias("m_away_probs"),
            pl.col("draw_prob").alias("m_draw_probs"),
            pl.col("home_prob").alias("m_home_probs"),
            pl.col("match_id")
        ])
    )

# Ensure match_id is present if it wasn't added in the 'else' block
if "match_id" not in val_probs.columns:
    val_probs = val_probs.with_columns(
        match_id = model_residual_df.filter(pl.col("Season").is_in(test_seasons)).get_column("match_id")
    )

val_probs

m_away_probs,m_draw_probs,m_home_probs,match_id
f64,f64,f64,u32
0.098115,0.187587,0.714298,17297
0.387791,0.291944,0.320266,17299
0.136052,0.2026,0.661348,17300
0.122144,0.208073,0.669784,17301
0.598639,0.221149,0.180213,17302
…,…,…,…
0.232668,0.287826,0.479507,23203
0.468479,0.279832,0.25169,23204
0.249868,0.338549,0.411582,23205


In [33]:
ev_series = (
    outcomes.filter(pl.col("Season").is_in(test_seasons))
    .join(
        val_probs,
        on='match_id'
    )
    .join(
        df.filter(pl.col("Season").is_in(test_seasons)).select("match_id", "MaxH", "MaxA", "MaxD"),
        on='match_id'
    )
    .with_columns(
        (pl.col("m_away_probs") * (pl.col("MaxA") - 1) - (1 - pl.col("m_away_probs"))).alias("EV_A"),
        (pl.col("m_home_probs") * (pl.col("MaxH") - 1) - (1 - pl.col("m_home_probs"))).alias("EV_H"),
        (pl.col("m_draw_probs") * (pl.col("MaxD") - 1) - (1 - pl.col("m_draw_probs"))).alias("EV_D")
    )
)

THRESHOLD = 0
MAX_THRESHOLD = 100

ev_series = ev_series.filter(
    pl.any_horizontal(
        pl.col("EV_A") >= THRESHOLD,
        pl.col("EV_H") >= THRESHOLD,
        pl.col("EV_D") >= THRESHOLD
    )
)

bet_series = (
    ev_series
    .with_columns(
        EV=pl.max_horizontal("EV_A", "EV_H", "EV_D")
    )
    .with_columns(
        odds=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.col("MaxA"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.col("MaxH"))
            .otherwise(pl.col("MaxD"))
        ),
        model_probability=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.col("m_away_probs"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.col("m_home_probs"))
            .otherwise(pl.col("m_draw_probs"))
        )
    )
    .with_columns(
        outcome = pl.col("outcome")
                .cast(pl.String)  # Move this here
                .replace({"0": "A", "1": "D", "2": "H"}) # Map strings to strings
    )
    .with_columns(
        model_predictions=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.lit("A"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.lit("H"))
            .otherwise(pl.lit("D"))
        )
    )
    .with_columns(
        outcome=pl.when(pl.col("outcome") == pl.col("model_predictions"))
        .then(pl.lit("win"))
        .otherwise(pl.lit("loss"))
    )
)

bet_series = bet_series.select("match_id", "Datetime", "Season", "outcome", "odds", "model_probability")
# Outlier
bet_series = bet_series.filter(pl.col("match_id") != 22149)
bet_series

match_id,Datetime,Season,outcome,odds,model_probability
u32,datetime[μs],str,str,f64,f64
17297,2024-07-26 19:45:00,"""2024/2025""","""loss""",1.42,0.714298
17299,2024-07-27 17:15:00,"""2024/2025""","""win""",3.56,0.291944
17300,2024-07-27 19:45:00,"""2024/2025""","""win""",1.55,0.661348
17301,2024-07-28 12:30:00,"""2024/2025""","""loss""",1.52,0.669784
17303,2024-07-28 17:30:00,"""2024/2025""","""win""",2.29,0.459711
…,…,…,…,…,…
23203,2026-03-09 17:00:00,"""2025/2026""","""win""",3.5,0.287826
23204,2026-03-09 17:00:00,"""2025/2026""","""loss""",3.7,0.279832
23205,2026-03-09 19:45:00,"""2025/2026""","""loss""",3.3,0.338549


In [34]:
from pipeline.risk import kelly_criterion, flat_bet, fixed_fraction
from pipeline.evaluation import evaluate_returns, get_evaluation_table, apply_clean_theme
from pipeline.metrics.performance import (
    median_PNL, mean_PNL, total_PNL, number_of_bets, win_rate, loss_rate, compound_return
)
from pipeline.metrics.risk_adjusted import sharpe_ratio, calmer_ratio, sortino_ratio
from pipeline.metrics.drawdown import (
    max_drawdown, max_time_underwater, average_drawdown, avg_time_underwater
)
from pipeline.metrics.volatility import volatility, downside_volatility, skewness, kurtosis, best_bet, worst_bet

KELLY_FRACTION = 0.2

metrics = {
    "Performance": [mean_PNL, median_PNL, total_PNL, number_of_bets, win_rate, loss_rate, compound_return],
    "Distribution": [volatility, downside_volatility, skewness, kurtosis, best_bet, worst_bet],
    "Risk Adjusted Returns": [sharpe_ratio, calmer_ratio, sortino_ratio],
    "Drawdown": [max_drawdown, max_time_underwater, average_drawdown, avg_time_underwater]
}

equity_series = kelly_criterion(bet_series, initial_bank_roll=500, fraction=KELLY_FRACTION)
px.line(
    equity_series,
    y="new_bank_roll",
    x="match_id",
)

In [35]:
match_id = equity_series.filter(pl.col("pnl") == pl.col("pnl").max()).get_column("match_id").item()
df.filter(pl.col("match_id") == match_id)

match_id,Datetime,League,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,home_mean_market_val,away_mean_market_val,home_team_elo,away_team_elo,home_team_elo_after,away_team_elo_after,Season,elo_diff,MV_log_diff,elo_per_value_diff,home_goals_ewm_4,home_goals_ewm_15,home_goals_ewm_30,home_sot_ratio_ewm_4,home_sot_ratio_ewm_15,home_sot_ratio_ewm_30,home_sot_a_ratio_ewm_4,home_sot_a_ratio_ewm_15,home_sot_a_ratio_ewm_30,home_conversion_rate_ewm_4,home_conversion_rate_ewm_15,home_conversion_rate_ewm_30,home_save_rate_ewm_4,home_save_rate_ewm_15,home_save_rate_ewm_30,home_elo_adv_ewm_4,away_goals_ewm_4,away_goals_ewm_15,away_goals_ewm_30,away_sot_ratio_ewm_4,away_sot_ratio_ewm_15,away_sot_ratio_ewm_30,away_sot_a_ratio_ewm_4,away_sot_a_ratio_ewm_15,away_sot_a_ratio_ewm_30,away_conversion_rate_ewm_4,away_conversion_rate_ewm_15,away_conversion_rate_ewm_30,away_save_rate_ewm_4,away_save_rate_ewm_15,away_save_rate_ewm_30,away_elo_adv_ewm_4,home_goals_momentum,away_goals_momentum,home_sot_ratio_momentum,away_sot_ratio_momentum,home_sot_a_ratio_momentum,away_sot_a_ratio_momentum,home_conversion_rate_momentum,away_conversion_rate_momentum,home_save_rate_momentum,away_save_rate_momentum,home_luck_factor,away_luck_factor
u32,datetime[μs],str,str,str,str,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
22372,2026-01-07 19:30:00,"""ENG""","""Everton""","""Wolves""","""D""",1,1,13,13,2,4,1.8,3.65,5.25,1.76,3.55,4.85,1.7636e7,1.3275e7,1801.654663,1661.810303,1796.483154,1666.981934,"""2025/2026""",139.84436,0.28408,6.656148,1.422258,1.14936,1.1373,0.355729,0.332718,0.337211,0.370017,0.342759,0.346794,0.323591,0.321381,0.320749,0.664333,0.696532,0.703685,-5.612438,1.671049,0.953642,0.947875,0.463575,0.379325,0.364853,0.296923,0.349092,0.356096,0.312829,0.293303,0.295843,0.678271,0.647329,0.642618,-165.1157,0.284958,0.723174,0.018518,0.098723,0.023222,-0.059173,0.002842,0.016986,-0.039352,0.035652,0.987924,0.991099


In [36]:
equity_series = equity_series.rename({"prev_bank_roll": "prev_bankroll", "new_bank_roll": "new_bankroll"})
returns = evaluate_returns(equity_series, metrics)
apply_clean_theme(get_evaluation_table(returns, title="Results", subtitle=""))


GT(_tbl_data=shape: (20, 6)
┌───────────────────────┬─────────────────────────────────┬─────────────┬──────────┬─────┬───────┐
│ Type                  ┆ Metric                          ┆ Value       ┆ fmt      ┆ dec ┆ sign  │
│ ---                   ┆ ---                             ┆ ---         ┆ ---      ┆ --- ┆ ---   │
│ str                   ┆ str                             ┆ f64         ┆ str      ┆ i64 ┆ bool  │
╞═══════════════════════╪═════════════════════════════════╪═════════════╪══════════╪═════╪═══════╡
│ Performance           ┆ Mean Pnl (Per Bet)              ┆ 0.452566    ┆ currency ┆ 2   ┆ true  │
│ Performance           ┆ Median Pnl (Per Bet)            ┆ -1.479471   ┆ currency ┆ 2   ┆ true  │
│ Performance           ┆ Total Pnl                       ┆ 2308.540216 ┆ currency ┆ 2   ┆ true  │
│ Performance           ┆ Number Of Bets                  ┆ 5101.0      ┆ number   ┆ 0   ┆ false │
│ Performance           ┆ Win Rate                        ┆ 0.418349    ┆ percent  ┆ 1   ┆ false │
│ …                     ┆ …                               ┆ …           ┆ …        ┆ …   ┆ …     │
│ Risk Adjusted Returns ┆ Sortino Ratio (Season-Adjusted… ┆ 2.883414    ┆ number   ┆ 3   ┆ false │
│ Drawdown              ┆ Max Drawdown                    ┆ 0.272483    ┆ percent  ┆ 1   ┆ false │
│ Drawdown              ┆ Max Time Underwater (Bets)      ┆ 1173.0      ┆ number   ┆ 0   ┆ false │
│ Drawdown              ┆ Average Drawdown                ┆ 0.090769    ┆ percent  ┆ 1   ┆ false │
│ Drawdown              ┆ Avg Time Underwater (Bets)      ┆ 36.313433   ┆ number   ┆ 1   ┆ false │
└───────────────────────┴─────────────────────────────────┴─────────────┴──────────┴─────┴───────┘, _body=<great_tables._gt_data.Body object at 0x0000026BA2A85FD0>, _boxhead=Boxhead([ColInfo(var='Type', type=<ColInfoTypeEnum.row_group: 3>, column_label='Type', column_align='left', column_width=None), ColInfo(var='Metric', type=<ColInfoTypeEnum.stub: 2>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Value', type=<ColInfoTypeEnum.default: 1>, column_label='Value', column_align='right', column_width=None), ColInfo(var='fmt', type=<ColInfoTypeEnum.hidden: 4>, column_label='fmt', column_align='left', column_width=None), ColInfo(var='dec', type=<ColInfoTypeEnum.hidden: 4>, column_label='dec', column_align='right', column_width=None), ColInfo(var='sign', type=<ColInfoTypeEnum.hidden: 4>, column_label='sign', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000026BA2A85A90>, _spanners=Spanners([]), _heading=Heading(title='Results', subtitle='', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000026BA2A86270>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000026BA2ADCCD0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Value', rownum=None, colnum=None, styles=[CellStyleFill(color='#2c3e50')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='fmt', rownum=None, colnum=None, styles=[CellStyleFill(color='#2c3e50')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='dec', rownum=None, colnum=None, styles=[CellStyleFill(color='#2c3e50')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='sign', rownum=None, colnum=None, styles=[CellStyleFill(color='#2c3e50')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='Value', rownum=None, colnum=None, styles=[CellStyleText(color='white', font=None, size='12px', align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform='uppercase', whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='fmt', rownum=None, colnum=None, styles=[CellStyleText(color='white', font=None, size='12px', align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=No

In [44]:
HOME_TEAM = "Fenerbahce"
AWAY_TEAM = "Gaziantep"

AVGH = 1.25
AVGD = 1 + (21/4)
AVGA = 1 + (11/1)

MAXH = 1.18
MAXA = 14.1
MAXD = 6.2

teams = df.select("match_id", "Datetime", "HomeTeam", "AwayTeam").unpivot(
    index=["match_id", "Datetime"], on=["HomeTeam", "AwayTeam"]
).sort("Datetime")

home_team_id = teams.filter(pl.col("value") == HOME_TEAM).tail(1).get_column("match_id").item()
away_team_id = teams.filter(pl.col("value") == AWAY_TEAM).tail(1).get_column("match_id").item()

home_df = df.filter(pl.col("match_id") == home_team_id)
away_df = df.filter(pl.col("match_id") == away_team_id)

home_features = latest_stats.filter(
    pl.col("match_id") == home_team_id,
    pl.col("team") == HOME_TEAM
)

away_features = latest_stats.filter(
    pl.col("match_id") == away_team_id,
    pl.col("team") == AWAY_TEAM
)

away_sot_ratio_momentum = away_features.select(
    pl.col(f"sot_ratio_ewm_{SHORT_WINDOW}") - pl.col(f"sot_ratio_ewm_{LONG_WINDOW}")
).item()

conversion_diff = (
    home_features.get_column("conversion_rate_ewm_30").item() 
    - away_features.get_column("conversion_rate_ewm_30").item() 
)

elo_per_value_diff = (
    home_df.select(pl.col("home_team_elo_after") / pl.col("home_mean_market_val").log()).item()
    - away_df.select(pl.col("away_team_elo_after") / pl.col("away_mean_market_val").log()).item()
)

pred_features = pl.DataFrame(
    {
        "elo_per_value_diff": (elo_per_value_diff, ),
        "conversion_diff": (conversion_diff, ),
        "away_sot_ratio_momentum": (away_sot_ratio_momentum, ),
        "AvgH": AVGH,
        "AvgD": AVGD,
        "AvgA": AVGA,
        "MaxH": MAXH,
        "MaxD": MAXD,
        "MaxA": MAXA,
    }
)

probs = pred_features.select(
    AvgH= 1 / pl.col("AvgH"),
    AvgA= 1 / pl.col("AvgA"),
    AvgD= 1 / pl.col("AvgD")
)

probs = probs.with_columns(implied_probs_sum=pl.col("AvgH") + pl.col("AvgD") + pl.col("AvgA"))
probs = probs.with_columns(
    pl.col(["AvgH", "AvgD", "AvgA"]) / pl.col("implied_probs_sum")
).rename({
    "AvgH": "home_prob",
    "AvgA": "away_prob",
    "AvgD": "draw_prob"
}).drop("implied_probs_sum")

pred_features

elo_per_value_diff,conversion_diff,away_sot_ratio_momentum,AvgH,AvgD,AvgA,MaxH,MaxD,MaxA
f64,f64,f64,f64,f64,f64,f64,f64,f64
10.51202,0.059478,0.006133,1.25,6.25,12.0,1.18,6.2,14.1


In [45]:
preds = final_model.predict(
    pred_features.select("elo_per_value_diff", "away_sot_ratio_momentum", "conversion_diff"), 
    market_probs=probs
)
val_probs = pl.DataFrame(preds, schema=["m_away_probs", "m_draw_probs", "m_home_probs"])
pred_df = pl.concat([pred_features, probs, val_probs], how="horizontal")
pred_df

elo_per_value_diff,conversion_diff,away_sot_ratio_momentum,AvgH,AvgD,AvgA,MaxH,MaxD,MaxA,home_prob,away_prob,draw_prob,m_away_probs,m_draw_probs,m_home_probs
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
10.51202,0.059478,0.006133,1.25,6.25,12.0,1.18,6.2,14.1,0.766773,0.079872,0.153355,0.070239,0.154881,0.77488


In [46]:
ev_series = (
    pred_df
    .with_columns(
        (pl.col("m_away_probs") * (pl.col("MaxA") - 1) - (1 - pl.col("m_away_probs"))).alias("EV_A"),
        (pl.col("m_home_probs") * (pl.col("MaxH") - 1) - (1 - pl.col("m_home_probs"))).alias("EV_H"),
        (pl.col("m_draw_probs") * (pl.col("MaxD") - 1) - (1 - pl.col("m_draw_probs"))).alias("EV_D")
    )
)

ev_series

elo_per_value_diff,conversion_diff,away_sot_ratio_momentum,AvgH,AvgD,AvgA,MaxH,MaxD,MaxA,home_prob,away_prob,draw_prob,m_away_probs,m_draw_probs,m_home_probs,EV_A,EV_H,EV_D
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
10.51202,0.059478,0.006133,1.25,6.25,12.0,1.18,6.2,14.1,0.766773,0.079872,0.153355,0.070239,0.154881,0.77488,-0.009634,-0.085641,-0.039738


In [47]:
ev_series = ev_series.filter(
    pl.any_horizontal(
        pl.col("EV_A") >= THRESHOLD,
        pl.col("EV_H") >= THRESHOLD,
        pl.col("EV_D") >= THRESHOLD
    )
)

bet_series = (
    ev_series
    .with_columns(
        EV=pl.max_horizontal("EV_A", "EV_H", "EV_D")
    )
    .with_columns(
        odds=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.col("MaxA"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.col("MaxH"))
            .otherwise(pl.col("MaxD"))
        ),
        model_probability=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.col("m_away_probs"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.col("m_home_probs"))
            .otherwise(pl.col("m_draw_probs"))
        )
    )
    .with_columns(
        model_predictions=(pl.when(pl.col("EV_A") == pl.col("EV"))
            .then(pl.lit("A"))
            .when(pl.col("EV_H") == pl.col("EV"))
            .then(pl.lit("H"))
            .otherwise(pl.lit("D"))
        )
    ).select("odds", "model_probability", "model_predictions", "EV")
)

bet_series

odds,model_probability,model_predictions,EV
f64,f64,str,f64


In [48]:
bet_series = (
    bet_series
    .with_columns(
        (pl.col('odds') - 1).alias('net_odds')
    )
    .with_columns(
        kelly_fraction=(
            (((pl.col('net_odds')*pl.col('model_probability') - (1 - pl.col('model_probability')))*KELLY_FRACTION)
            /pl.col('net_odds'))
        )
    )
)

bet_series

odds,model_probability,model_predictions,EV,net_odds,kelly_fraction
f64,f64,str,f64,f64,f64
